# 48h PM10 模型：与 `monthtail_2` 测试日对齐的按预报时效（lead）评估

**目的**
- 从 **`s2_data_monthtail_v2.ipynb`** 构建的 **`ml_dataset_s2_tianji_12h_pm10_monthtail_2`** 的 `meta_test.csv` 提取测试集日历日（与当前 S2 训练划分一致）。
- 在 **`PMST_s2_data_48h_pm10.ipynb`** 产出的 48h 特征数据上进行推理，并按 `lead_hour` 统计模型指标随 lead 的变化。
- 读取 IFS 站点产品的 **0-48h 全部逐小时 lead**，与模型按同一条 lead 轴做对比，输出同图曲线。

**说明**
- `ml_dataset_fe_12h_48h_pm10` 若用 `PMST_s2_data` 的比例划分构建，则与 monthtail 测试日不完全一致；优先使用与 monthtail 同逻辑的 **`ml_dataset_fe_12h_48h_pm10_testonly`**。
- IFS 需要使用新的逐小时站点文件：**`IFS_VIS_0_48h_stations_2025_00_12.nc`**。

In [14]:
# ========= 路径与超参（按你的环境修改） =========
import os
import sys

try:
    _file = __file__
except NameError:
    _file = os.path.join(os.getcwd(), "paper_eval", "eval_48h_pm10_lead_curves_monthtail_test.ipynb")

BASE = os.path.dirname(os.path.dirname(os.path.abspath(_file)))
if not os.path.exists(os.path.join(BASE, "paper_eval")):
    BASE = os.path.dirname(BASE)

PAPER_EVAL_DIR = os.path.join(BASE, "paper_eval")
TRAIN_DIR = os.path.join(BASE, "train")
sys.path.insert(0, BASE)
sys.path.insert(0, TRAIN_DIR)
os.chdir(BASE)

# 12h monthtail：用于定义「测试日」集合
meta_test_12h_path = os.path.join(BASE, "ml_dataset_s2_tianji_12h_pm10_monthtail_2", "meta_test.csv")

# 48h 数据集：优先 testonly（与 monthtail 同划分）；否则为含 train/val/test 的目录并配合测试日过滤
data_48h_dir = os.path.join(BASE, "ml_dataset_fe_12h_48h_pm10_testonly")
if not os.path.isdir(data_48h_dir):
    data_48h_dir = os.path.join(BASE, "ml_dataset_fe_12h_48h_pm10")

exp_id = "exp_1773802782_pm10_more_test"
ckpt_path = os.path.join(BASE, "checkpoints", f"{exp_id}_S2_PhaseB_best_score.pt")
scaler_path = os.path.join(BASE, "checkpoints", "robust_scaler_w12_dyn26_s2_48h_pm10.pkl")
season_th_path = os.path.join(BASE, "checkpoints", f"{exp_id}_season_thresholds.pt")

output_dir = os.path.join(BASE, "paper_eval_results_48h_lead_monthtail_test")
os.makedirs(output_dir, exist_ok=True)

fog_th, mist_th = 0.10, 0.30
no_calibration = True
threshold_rule = "mutual"
use_cpu = False
batch_size = 4096

window_size = 12
dyn_vars_count = 26
# 修正后的 48h testonly 数据：extra = 32(fog_fe) + 4(cyclical) = 36；lead_hour 仅保存在 meta 中
extra_feat_dim_expected = 36

ifs_all_leads_nc = os.path.join(BASE, "IFS_VIS_0_48h_stations_2025_00_12.nc")
run_ifs_baseline = os.path.isfile(ifs_all_leads_nc)

print("BASE:", BASE)
print("12h meta_test:", meta_test_12h_path)
print("48h data_dir:", data_48h_dir)
print("ckpt:", ckpt_path)
print("scaler:", scaler_path)
print("output_dir:", output_dir)
print("IFS 0-48h baseline:", run_ifs_baseline, ifs_all_leads_nc if run_ifs_baseline else "")

BASE: /public/home/putianshu/vis_mlp
12h meta_test: /public/home/putianshu/vis_mlp/ml_dataset_s2_tianji_12h_pm10_monthtail_2/meta_test.csv
48h data_dir: /public/home/putianshu/vis_mlp/ml_dataset_fe_12h_48h_pm10_testonly
ckpt: /public/home/putianshu/vis_mlp/checkpoints/exp_1773802782_pm10_more_test_S2_PhaseB_best_score.pt
scaler: /public/home/putianshu/vis_mlp/checkpoints/robust_scaler_w12_dyn26_s2_48h_pm10.pkl
output_dir: /public/home/putianshu/vis_mlp/paper_eval_results_48h_lead_monthtail_test
IFS 0-48h baseline: True /public/home/putianshu/vis_mlp/IFS_VIS_0_48h_stations_2025_00_12.nc


In [15]:
# ========= paper_eval.metrics_core + 动态导入 =========
import types
import importlib.util
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

for _name in list(sys.modules.keys()):
    if _name == "paper_eval" or _name.startswith("paper_eval."):
        del sys.modules[_name]

paper_eval_pkg = types.ModuleType("paper_eval")
paper_eval_pkg.__path__ = [PAPER_EVAL_DIR]
paper_eval_pkg.__file__ = os.path.join(PAPER_EVAL_DIR, "__init__.py")
sys.modules["paper_eval"] = paper_eval_pkg


def _load_sub(name):
    full = f"paper_eval.{name}"
    path = os.path.join(PAPER_EVAL_DIR, f"{name}.py")
    spec = importlib.util.spec_from_file_location(full, path)
    mod = importlib.util.module_from_spec(spec)
    sys.modules[full] = mod
    spec.loader.exec_module(mod)
    return mod


plot_style_mod = _load_sub("plot_style")
metrics_core_mod = _load_sub("metrics_core")
setup_paper_style = plot_style_mod.setup_paper_style
save_figure = plot_style_mod.save_figure

pred_from_thresholds = metrics_core_mod.pred_from_thresholds
pred_from_thresholds_mutual = getattr(metrics_core_mod, "pred_from_thresholds_mutual", pred_from_thresholds)
pred_from_joint_thresholds = getattr(metrics_core_mod, "pred_from_joint_thresholds", pred_from_thresholds)
compute_rare_event_report = metrics_core_mod.compute_rare_event_report

if hasattr(metrics_core_mod, "pred_from_season_thresholds"):
    pred_from_season_thresholds = metrics_core_mod.pred_from_season_thresholds
else:
    def pred_from_season_thresholds(
        probs, months, season_thresholds, default_fog_th=0.46, default_mist_th=0.38, rule="default"
    ):
        month_to_season = {12: "DJF", 1: "DJF", 2: "DJF", 3: "MAM", 4: "MAM", 5: "MAM", 6: "JJA", 7: "JJA", 8: "JJA", 9: "SON", 10: "SON", 11: "SON"}
        months = np.asarray(months, dtype=np.int32).ravel()
        n = probs.shape[0]
        fog_th_arr = np.full(n, default_fog_th, dtype=np.float64)
        mist_th_arr = np.full(n, default_mist_th, dtype=np.float64)
        for i in range(n):
            season = month_to_season.get(int(months[i]))
            if season and season in season_thresholds:
                fog_th_arr[i] = season_thresholds[season]["fog_th"]
                mist_th_arr[i] = season_thresholds[season]["mist_th"]
        if rule == "mutual":
            return pred_from_thresholds_mutual(probs, fog_th_arr, mist_th_arr)
        if rule == "joint":
            return pred_from_joint_thresholds(probs, fog_th_arr, mist_th_arr)
        return pred_from_thresholds(probs, fog_th_arr, mist_th_arr)

from sklearn.metrics import average_precision_score

print("paper_eval metrics loaded.")

paper_eval metrics loaded.


In [16]:
# ========= 测试日、lead 解析、数据加载、推理（与 PMST_net_test_11_s2_pm10 / paper_eval 一致） =========


def load_test_calendar_dates_from_12h(meta_csv_path):
    """与训练用 monthtail 测试集一致的 UTC 日历日集合。"""
    m = pd.read_csv(meta_csv_path, parse_dates=["time"])
    dates = pd.to_datetime(m["time"]).dt.normalize().dt.date.unique()
    return set(dates), m


def infer_extra_feat_dim(n_cols, window_size, dyn_vars_count):
    """n_cols = win*dyn + 5(static_cont) + 1(veg) + extra"""
    base = window_size * dyn_vars_count + 5 + 1
    return int(n_cols - base)


def normalize_station_id(values):
    """统一 station_id 表示，避免 '12345' / '12345.0' / '0012345' 不匹配。"""
    s = pd.Series(values)
    s = s.astype(str).str.strip()
    s = s.str.replace(r"\.0$", "", regex=True)
    s = s.str.replace(r"^0+(\d+)$", r"\1", regex=True)
    return s.fillna("").values


def load_48h_xy_meta(data_dir, window_size, dyn_vars_count):
    Xp = os.path.join(data_dir, "X_test.npy")
    yp = os.path.join(data_dir, "y_test.npy")
    mp = os.path.join(data_dir, "meta_test.csv")
    if not all(os.path.exists(p) for p in (Xp, yp, mp)):
        raise FileNotFoundError(f"需要 X_test.npy / y_test.npy / meta_test.csv under {data_dir}")
    X = np.load(Xp, mmap_mode="r")
    y_raw = np.load(yp, mmap_mode="r")
    meta = pd.read_csv(mp, parse_dates=["time"])
    ex = infer_extra_feat_dim(X.shape[1], window_size, dyn_vars_count)
    return X, y_raw, meta, ex


def get_lead_hours(meta, X, window_size, dyn_vars_count, extra_feat_dim):
    """优先使用 meta_test.csv 中显式保存的 lead_hour；只有旧数据才回退到 X 推断。"""
    if "lead_hour" in meta.columns:
        lead = pd.to_numeric(meta["lead_hour"], errors="coerce").values.astype(np.float32)
        if np.isfinite(lead).any():
            return lead

    split_dyn = window_size * dyn_vars_count
    off = split_dyn + 6
    li = max(0, extra_feat_dim - 2)
    lead_norm = np.asarray(X[:, off + li], dtype=np.float64)
    return np.clip(lead_norm * 48.0, 0.0, 48.0).astype(np.float32)


def y_raw_to_cls(y_raw):
    y_raw = np.asarray(y_raw, dtype=np.float64)
    if np.max(y_raw) < 100:
        y_raw = y_raw * 1000.0
    y_cls = np.zeros(len(y_raw), dtype=np.int64)
    y_cls[y_raw >= 500] = 1
    y_cls[y_raw >= 1000] = 2
    return y_cls, y_raw


def _regime_features_from_meta(meta_slice):
    month = np.asarray(meta_slice["month"].values, dtype=np.int32)
    hour = np.asarray(meta_slice["hour"].values, dtype=np.int32)
    lats = meta_slice["lat"].values if "lat" in meta_slice.columns else np.zeros(len(month))
    lons = meta_slice["lon"].values if "lon" in meta_slice.columns else np.zeros(len(month))
    is_coastal = ((lons >= 118) & (lats >= 18) & (lats <= 42)).astype(np.float32)
    is_day = ((hour >= 6) & (hour < 18)).astype(np.float32)
    djf = ((month == 12) | (month <= 2)).astype(np.float32)
    mam = ((month >= 3) & (month <= 5)).astype(np.float32)
    jja = ((month >= 6) & (month <= 8)).astype(np.float32)
    son = ((month >= 9) & (month <= 11)).astype(np.float32)
    return np.column_stack([djf, mam, jja, son, is_day, is_coastal]).astype(np.float32)


def run_inference_48h(
    X,
    scaler,
    model,
    device,
    batch_size,
    window_size,
    dyn_vars_count,
    extra_feat_dim,
    temperature,
    meta,
):
    import torch
    import torch.nn.functional as F

    T = 1.0 if temperature is None or temperature == 1.0 else float(temperature)
    N = len(X)
    split_dyn = int(dyn_vars_count) * window_size
    log_mask = np.zeros(split_dyn, dtype=bool)
    pm10_dyn_index = int(dyn_vars_count) - 1
    for t in range(window_size):
        base = t * int(dyn_vars_count)
        for i in [2, 4, 9, pm10_dyn_index]:
            log_mask[base + i] = True

    use_cuda = device.type == "cuda"
    non_blocking = use_cuda
    need_regime = getattr(model, "regime_feat_dim", 0)
    if need_regime and meta is None:
        raise ValueError("model expects regime features")

    all_probs = []
    for start in range(0, N, batch_size):
        end = min(start + batch_size, N)
        rows = np.asarray(X[start:end], dtype=np.float32)
        feats = rows[:, : split_dyn + 5]
        feats[:, :split_dyn] = np.where(
            log_mask,
            np.log1p(np.maximum(feats[:, :split_dyn], 0)),
            feats[:, :split_dyn],
        )
        if scaler is not None:
            feats = (feats - scaler.center_) / (scaler.scale_ + 1e-6)
        veg = rows[:, split_dyn + 5 : split_dyn + 6]
        extra = rows[:, split_dyn + 6 :]
        if extra.shape[1] < extra_feat_dim:
            extra = np.pad(extra, ((0, 0), (0, extra_feat_dim - extra.shape[1])), mode="constant")
        elif extra.shape[1] > extra_feat_dim:
            extra = extra[:, :extra_feat_dim]
        final = np.concatenate([np.clip(feats, -10, 10), veg, np.clip(extra, -10, 10)], axis=1)
        if need_regime and meta is not None:
            regime = _regime_features_from_meta(meta.iloc[start:end])
            final = np.concatenate([final, regime], axis=1)
        final = np.nan_to_num(final, nan=0.0)
        x = torch.from_numpy(final).float().to(device, non_blocking=non_blocking)
        with torch.inference_mode():
            fine, _, _ = model(x)
            probs = F.softmax(fine / T, dim=1)
        all_probs.append(probs.cpu().numpy())
    return np.concatenate(all_probs, axis=0)


def pred_from_rule(probs, months, season_thresholds, fog_th, mist_th, rule):
    if season_thresholds is not None:
        return pred_from_season_thresholds(probs, months, season_thresholds, fog_th, mist_th, rule=rule)
    if rule == "mutual":
        return pred_from_thresholds_mutual(probs, fog_th, mist_th)
    if rule == "joint":
        return pred_from_joint_thresholds(probs, fog_th, mist_th)
    return pred_from_thresholds(probs, fog_th, mist_th)


def pr_auc_ovr(probs, y_cls, c):
    y = (y_cls == c).astype(np.int32)
    if y.sum() == 0 or (1 - y).sum() == 0:
        return float("nan")
    return float(average_precision_score(y, probs[:, c]))


print("helpers OK.")

helpers OK.


In [17]:
# ========= 解析测试日、过滤 48h 样本、加载模型 =========
import torch

is_testonly_dataset = os.path.basename(data_48h_dir).endswith("testonly")

test_dates, meta_12h_ref = load_test_calendar_dates_from_12h(meta_test_12h_path)
pd.DataFrame(sorted(test_dates), columns=["test_calendar_date_utc"]).to_csv(
    os.path.join(output_dir, "monthtail_test_calendar_dates.csv"), index=False
)
print(f"[12h monthtail] unique test calendar days: {len(test_dates)}")

X, y_raw, meta, extra_dim = load_48h_xy_meta(data_48h_dir, window_size, dyn_vars_count)
print(f"[48h] X shape={X.shape}, inferred extra_feat_dim={extra_dim} (expected ~36)")
if extra_dim != extra_feat_dim_expected:
    print(f"[WARN] extra dim mismatch: got {extra_dim}, expected {extra_feat_dim_expected}")

meta = meta.copy()
meta["time"] = pd.to_datetime(meta["time"])
meta["month"] = meta["time"].dt.month
meta["hour"] = meta["time"].dt.hour
meta["date_utc"] = meta["time"].dt.date
meta["station_id_norm"] = normalize_station_id(meta["station_id"])
meta["sample_key"] = (
    meta["time"].dt.strftime("%Y-%m-%d %H:%M:%S")
    + "__"
    + pd.Series(meta["station_id_norm"], dtype="string").fillna("")
)

meta_12h_ref = meta_12h_ref.copy()
meta_12h_ref["time"] = pd.to_datetime(meta_12h_ref["time"])
meta_12h_ref["station_id_norm"] = normalize_station_id(meta_12h_ref["station_id"])
meta_12h_ref["sample_key"] = (
    meta_12h_ref["time"].dt.strftime("%Y-%m-%d %H:%M:%S")
    + "__"
    + pd.Series(meta_12h_ref["station_id_norm"], dtype="string").fillna("")
)
ref_keys = set(meta_12h_ref["sample_key"].tolist())

mask_day = meta["date_utc"].isin(test_dates)
mask_key = meta["sample_key"].isin(ref_keys)

if is_testonly_dataset:
    idx = np.where(mask_key.values)[0]
    print(f"[filter] testonly exact key overlap: {mask_key.mean():.2%} ({len(idx)} / {len(meta)})")
    if len(idx) == 0:
        idx = np.where(mask_day.values)[0]
        print(f"[WARN] exact key overlap is zero; fallback to test-day filter: {len(idx)} / {len(meta)}")
else:
    idx = np.where(mask_key.values)[0]
    print(f"[filter] exact monthtail sample overlap: {len(idx)} / {len(meta)}")
    if len(idx) == 0:
        idx = np.where(mask_day.values)[0]
        print(f"[WARN] exact key overlap is zero; fallback to test-day filter: {len(idx)} / {len(meta)}")

if len(idx) == 0:
    raise RuntimeError("无样本落在测试集内：请检查 48h 数据目录、station_id 归一化或 UTC 时间对齐。")

X_sub = X[idx]
y_raw_sub = np.asarray(y_raw[idx])
meta_sub = meta.iloc[idx].reset_index(drop=True)
y_cls, y_raw_sub = y_raw_to_cls(y_raw_sub)

lead_h = get_lead_hours(meta_sub, X_sub, window_size, dyn_vars_count, extra_dim)
meta_sub["lead_hour"] = lead_h.astype(np.float32)
lead_hi = np.rint(lead_h).astype(np.int32)

n_dup = int(meta_sub["sample_key"].duplicated().sum())
print(f"[align] duplicate (time, station) keys in filtered 48h set: {n_dup}")
print(f"[lead] min/max (h): {lead_h.min():.2f} / {lead_h.max():.2f}; unique integer hours: {np.unique(lead_hi).size}")
print("[lead] first 20 unique hours:", sorted(np.unique(lead_hi))[:20])

align_diag = pd.DataFrame(
    {
        "metric": ["n_total_48h", "n_test_dates_48h", "n_exact_key_overlap", "n_filtered", "duplicate_time_station_keys"],
        "value": [len(meta), int(mask_day.sum()), int(mask_key.sum()), len(meta_sub), n_dup],
    }
)
align_diag.to_csv(os.path.join(output_dir, "alignment_diagnostics.csv"), index=False)

scaler = joblib.load(scaler_path) if os.path.exists(scaler_path) else None

if use_cpu or os.environ.get("CUDA_VISIBLE_DEVICES") == "":
    device = torch.device("cpu")
else:
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

from PMST_net_test_11_s2_pm10 import ImprovedDualStreamPMSTNet

model = ImprovedDualStreamPMSTNet(
    dyn_vars_count=dyn_vars_count,
    window_size=window_size,
    hidden_dim=512,
    dropout=0.3,
    extra_feat_dim=extra_dim,
    num_classes=3,
).to(device)

state = torch.load(ckpt_path, map_location=device)
state = {k.replace("module.", ""): v for k, v in state.items()}
model.load_state_dict(state, strict=False)
model.eval()
if device.type == "cuda" and torch.cuda.device_count() > 1:
    model = torch.nn.DataParallel(model)

if "monthtail_2" in ckpt_path and "48h" not in os.path.basename(ckpt_path):
    print("[WARN] 当前 checkpoint 名称看起来不是专门的 48h 预报模型；若性能显著劣化，这更可能是输入分布不匹配，而非评估代码问题。")

season_thresholds = None
temperature = None
if (not no_calibration) and os.path.exists(season_th_path):
    try:
        try:
            cal = torch.load(season_th_path, map_location="cpu", weights_only=True)
        except TypeError:
            cal = torch.load(season_th_path, map_location="cpu")
        season_thresholds = cal.get("season_thresholds")
        temperature = cal.get("temperature")
        if temperature is not None:
            temperature = float(temperature)
        print("Calibration:", season_th_path, "T=", temperature)
    except Exception as e:
        print("[WARN] calibration load failed:", e)

probs = run_inference_48h(
    X_sub,
    scaler,
    model,
    device,
    batch_size,
    window_size,
    dyn_vars_count,
    extra_dim,
    temperature,
    meta_sub,
)

preds = pred_from_rule(
    probs,
    meta_sub["month"].values,
    season_thresholds,
    fog_th,
    mist_th,
    threshold_rule,
)

report_full = compute_rare_event_report(probs, y_cls, fog_th, mist_th, pred=preds)
print("--- Full subset (evaluation dataset, filtered) ---")
for k, v in report_full.items():
    if isinstance(v, (int, float)) and k in (
        "Fog_CSI", "Fog_R", "Fog_P", "Mist_CSI", "Mist_P", "Mist_R", "low_vis_precision", "false_positive_rate"
    ):
        print(f"  {k}: {v:.4f}")

[12h monthtail] unique test calendar days: 36
[48h] X shape=(1899729, 354), inferred extra_feat_dim=36 (expected ~36)
[filter] using full testonly dataset, overlap with 12h test dates = 98.76%
[lead] min/max (h): 12.00 / 24.00; unique integer hours: 13
[lead] first 20 unique hours: [12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24]
--- Full subset (evaluation dataset, all leads) ---
  Fog_P: 0.0172
  Fog_R: 0.3747
  Mist_P: 0.0081
  Mist_R: 0.5048
  Fog_CSI: 0.0168
  Mist_CSI: 0.0080
  low_vis_precision: 0.0228
  false_positive_rate: 0.7796


In [ ]:
# ========= 按整数 lead（起报后小时）聚合指标 =========
rows = []
for h in sorted(np.unique(lead_hi)):
    m = lead_hi == h
    if m.sum() < 50:
        continue
    ph, yh = probs[m], y_cls[m]
    pr = preds[m]
    rep = compute_rare_event_report(ph, yh, fog_th, mist_th, pred=pr)
    row = {
        "lead_hour": int(h),
        "n": int(m.sum()),
        "n_fog": int((yh == 0).sum()),
        "n_mist": int((yh == 1).sum()),
        "n_clear": int((yh == 2).sum()),
    }
    for key in (
        "Fog_CSI",
        "Fog_POD",
        "Fog_FAR",
        "Fog_R",
        "Fog_P",
        "Mist_CSI",
        "Mist_POD",
        "Mist_FAR",
        "Mist_R",
        "Mist_P",
        "low_vis_precision",
        "false_positive_rate",
        "macro_f1",
    ):
        if key in rep:
            row[key] = rep[key]
    row["PR_AUC_Fog"] = pr_auc_ovr(ph, yh, 0)
    row["PR_AUC_Mist"] = pr_auc_ovr(ph, yh, 1)
    rows.append(row)

lead_df = pd.DataFrame(rows).sort_values("lead_hour").reset_index(drop=True)
out_csv = os.path.join(output_dir, "metrics_by_lead_hour.csv")
lead_df.to_csv(out_csv, index=False)
print("Saved:", out_csv)
try:
    from IPython.display import display
except ImportError:
    display = print
display(lead_df)
print("[sanity] total by lead:", int(lead_df["n"].sum()), " / subset size:", len(meta_sub))

Saved: /public/home/putianshu/vis_mlp/paper_eval_results_48h_lead_monthtail_test/metrics_by_lead_hour.csv


,lead_hour,n,n_fog,n_mist,n_clear,Fog_CSI,Fog_POD,Fog_FAR,Fog_R,Fog_P,Mist_CSI,Mist_POD,Mist_FAR,Mist_R,Mist_P,low_vis_precision,false_positive_rate,macro_f1,PR_AUC_Fog,PR_AUC_Mist
0,12,145990,2052,1310,142628,0.018713,0.358674,0.980639,0.358674,0.019361,0.008499,0.496947,0.991428,0.496947,0.008572,0.025659,0.778473,0.138463,0.023383,0.008753
1,13,147218,2275,1390,143553,0.020761,0.365714,0.978463,0.365714,0.021537,0.009161,0.508633,0.990757,0.508633,0.009243,0.027397,0.779977,0.139490,0.025913,0.009247
2,14,147650,2561,1541,143548,0.022547,0.359625,0.976510,0.359625,0.023490,0.010369,0.524335,0.989532,0.524335,0.010468,0.030757,0.785918,0.138746,0.027280,0.010211
3,15,141021,2476,1602,136943,0.022488,0.350969,0.976536,0.350969,0.023464,0.011272,0.530587,0.988614,0.530587,0.011386,0.032026,0.789489,0.137679,0.026619,0.011351
4,16,146577,2193,1377,143007,0.020265,0.364797,0.978994,0.364797,0.021006,0.008931,0.509078,0.990991,0.509078,0.009009,0.027136,0.788430,0.135275,0.024891,0.009098
5,17,141090,1578,1177,138335,0.016717,0.394804,0.982844,0.394804,0.017156,0.007735,0.495327,0.992203,0.495327,0.007797,0.021839,0.785499,0.133593,0.021531,0.007793
6,18,145786,1114,1046,143626,0.012289,0.421005,0.987500,0.421005,0.012500,0.006656,0.492352,0.993298,0.492352,0.006702,0.016587,0.783055,0.131170,0.020065,0.007015
7,19,146029,997,895,144137,0.011476,0.437312,0.988352,0.437312,0.011648,0.005759,0.493855,0.994207,0.493855,0.005793,0.014834,0.777323,0.132655,0.019879,0.005824
8,20,146941,1073,959,144909,0.011525,0.411929,0.988283,0.411929,0.011717,0.006213,0.495308,0.993748,0.495308,0.006252,0.015973,0.772057,0.135315,0.021973,0.006333
9,21,141092,1210,1042,138840,0.012449,0.377686,0.987290,0.377686,0.012710,0.006809,0.482726,0.993141,0.482726,0.006859,0.018391,0.772688,0.136005,0.019939,0.007074


[sanity] total by lead: 1899729  / subset size: 1899729


In [19]:
# ========= 曲线图（IFS 48h 对比点会在下一格生成后叠加） =========
setup_paper_style()
ifs_point = None
if lead_df.empty:
    print("lead_df 为空：样本过少或 lead 分箱无足够计数。")
else:
    fig, axes = plt.subplots(2, 2, figsize=(11, 8), sharex=True)
    x = lead_df["lead_hour"].values

    axes[0, 0].plot(x, lead_df["Fog_CSI"], "o-", label="Model Fog CSI")
    axes[0, 0].plot(x, lead_df["Mist_CSI"], "s-", label="Model Mist CSI")
    axes[0, 0].set_ylabel("CSI")
    axes[0, 0].legend(fontsize=8)
    axes[0, 0].grid(True, alpha=0.3)

    axes[0, 1].plot(x, lead_df["Fog_R"], "o-", label="Model Fog Recall")
    axes[0, 1].plot(x, lead_df["Fog_P"], "^-", label="Model Fog Precision")
    axes[0, 1].plot(x, lead_df["Mist_R"], "s-", label="Model Mist Recall")
    axes[0, 1].plot(x, lead_df["Mist_P"], "d-", label="Model Mist Precision")
    axes[0, 1].set_ylabel("Precision / Recall")
    axes[0, 1].legend(fontsize=8)
    axes[0, 1].grid(True, alpha=0.3)

    axes[1, 0].plot(x, lead_df["PR_AUC_Fog"], "o-", label="Model PR-AUC Fog")
    axes[1, 0].plot(x, lead_df["PR_AUC_Mist"], "s-", label="Model PR-AUC Mist")
    axes[1, 0].set_ylabel("PR-AUC")
    axes[1, 0].set_xlabel("Lead time (h)")
    axes[1, 0].legend(fontsize=8)
    axes[1, 0].grid(True, alpha=0.3)

    axes[1, 1].plot(x, lead_df["low_vis_precision"], "o-", label="Model Low-vis precision")
    axes[1, 1].plot(x, lead_df["false_positive_rate"], "s-", label="Model FPR (Clear)")
    axes[1, 1].set_ylabel("Rate")
    axes[1, 1].set_xlabel("Lead time (h)")
    axes[1, 1].legend(fontsize=8)
    axes[1, 1].grid(True, alpha=0.3)

    fig.suptitle("48h PM10 model — metrics vs lead")
    fig.tight_layout()
    fig_path = os.path.join(output_dir, "fig_metrics_vs_lead_hour_model_only.png")
    save_figure(fig, fig_path)
    plt.show()

  [Fig] Saved → /public/home/putianshu/vis_mlp/paper_eval_results_48h_lead_monthtail_test/fig_metrics_vs_lead_hour_model_only.png


In [20]:
# ========= IFS 0-48h：与模型按同一 lead 轴逐小时对比 =========
import xarray as xr
from sklearn.metrics import classification_report, cohen_kappa_score


def _vis_to_cls_scalar(vi):
    if not np.isfinite(vi):
        return -1
    if vi < 500:
        return 0
    if vi < 1000:
        return 1
    return 2


ifs_df = pd.DataFrame()

if not run_ifs_baseline:
    print("跳过 IFS：未找到", ifs_all_leads_nc)
else:
    ds_ifs = xr.open_dataset(ifs_all_leads_nc)
    stations_ifs = normalize_station_id(ds_ifs["station"].values)
    lead_vals_ifs = ds_ifs["lead_hour"].values.astype(np.int32)
    valid_time_grid = pd.to_datetime(ds_ifs["valid_time"].values)
    vis_ifs = ds_ifs["VIS_ifs"].values
    n_rec, n_lead, n_st = vis_ifs.shape

    if valid_time_grid.shape != (n_rec, n_lead):
        raise ValueError(
            f"IFS valid_time shape mismatch: got {valid_time_grid.shape}, expected {(n_rec, n_lead)}"
        )

    valid_time_ifs = np.repeat(valid_time_grid.reshape(-1), n_st)
    lead_idx = np.tile(np.repeat(np.arange(n_lead), n_st), n_rec)
    st_idx = np.tile(np.arange(n_st), n_rec * n_lead)

    df_right = pd.DataFrame(
        {
            "t": valid_time_ifs,
            "lead_hour": lead_vals_ifs[lead_idx],
            "station_id_norm": stations_ifs[st_idx],
            "vis_ifs": vis_ifs.reshape(-1).astype(np.float64),
        }
    )
    df_right = df_right.sort_values("t")

    df_left = meta_sub[["time", "lead_hour", "station_id_norm"]].copy()
    df_left = df_left.rename(columns={"time": "t"})
    df_left["lead_hour"] = np.rint(df_left["lead_hour"]).astype(np.int32)
    df_left["obs_cls"] = y_cls
    df_left = df_left.sort_values("t")

    merged_parts = []
    for (st, lead_h), L in df_left.groupby(["station_id_norm", "lead_hour"]):
        R = df_right[(df_right["station_id_norm"] == st) & (df_right["lead_hour"] == lead_h)].sort_values("t")
        if R.empty:
            continue
        merged_parts.append(
            pd.merge_asof(L.sort_values("t"), R, on="t", tolerance=pd.Timedelta("1h"), direction="nearest")
        )

    merged = pd.concat(merged_parts, axis=0, ignore_index=True) if merged_parts else pd.DataFrame()
    ds_ifs.close()

    if len(merged) == 0:
        print("[IFS] merge 无结果")
    else:
        merged["ifs_cls"] = merged["vis_ifs"].map(lambda v: _vis_to_cls_scalar(v) if pd.notna(v) else -1)
        ok_all = merged["ifs_cls"] >= 0
        merged_ok = merged.loc[ok_all].copy()
        print(f"[IFS] usable {len(merged_ok)} / {len(merged)} across all leads")

        rows = []
        for h, sub in merged_ok.groupby("lead_hour"):
            if len(sub) < 50:
                continue
            y_t = sub["obs_cls"].values.astype(np.int64)
            y_p = sub["ifs_cls"].values.astype(np.int64)
            rep = compute_rare_event_report(
                np.full((len(sub), 3), 1.0 / 3.0),
                y_t,
                fog_th,
                mist_th,
                pred=y_p,
            )
            row = {"lead_hour": int(h), "n": int(len(sub))}
            for key in (
                "Fog_CSI",
                "Fog_R",
                "Fog_P",
                "Mist_CSI",
                "Mist_R",
                "Mist_P",
                "low_vis_precision",
                "false_positive_rate",
            ):
                row[key] = rep.get(key, np.nan)
            rows.append(row)

        ifs_df = pd.DataFrame(rows).sort_values("lead_hour").reset_index(drop=True)
        if not ifs_df.empty:
            ifs_df.to_csv(os.path.join(output_dir, "ifs_metrics_by_lead_hour.csv"), index=False)
            lead48 = merged_ok[np.rint(merged_ok["lead_hour"]).astype(int) == 48]
            if len(lead48) > 100:
                print("Cohen kappa (IFS vs obs @48h):", cohen_kappa_score(lead48["obs_cls"].values, lead48["ifs_cls"].values))
                print(classification_report(lead48["obs_cls"].values, lead48["ifs_cls"].values, target_names=["Fog", "Mist", "Clear"], zero_division=0))

if (not lead_df.empty) and (not ifs_df.empty):
    fig, axes = plt.subplots(2, 2, figsize=(11, 8), sharex=True)
    x = lead_df["lead_hour"].values
    xi = ifs_df["lead_hour"].values

    axes[0, 0].plot(x, lead_df["Fog_CSI"], "o-", label="Model Fog CSI")
    axes[0, 0].plot(x, lead_df["Mist_CSI"], "s-", label="Model Mist CSI")
    axes[0, 0].plot(xi, ifs_df["Fog_CSI"], "o--", label="IFS Fog CSI")
    axes[0, 0].plot(xi, ifs_df["Mist_CSI"], "s--", label="IFS Mist CSI")
    axes[0, 0].set_ylabel("CSI")
    axes[0, 0].legend(fontsize=8)
    axes[0, 0].grid(True, alpha=0.3)

    axes[0, 1].plot(x, lead_df["Fog_R"], "o-", label="Model Fog Recall")
    axes[0, 1].plot(x, lead_df["Fog_P"], "^-", label="Model Fog Precision")
    axes[0, 1].plot(x, lead_df["Mist_R"], "s-", label="Model Mist Recall")
    axes[0, 1].plot(x, lead_df["Mist_P"], "d-", label="Model Mist Precision")
    axes[0, 1].plot(xi, ifs_df["Fog_R"], "o--", label="IFS Fog Recall")
    axes[0, 1].plot(xi, ifs_df["Fog_P"], "^--", label="IFS Fog Precision")
    axes[0, 1].plot(xi, ifs_df["Mist_R"], "s--", label="IFS Mist Recall")
    axes[0, 1].plot(xi, ifs_df["Mist_P"], "d--", label="IFS Mist Precision")
    axes[0, 1].set_ylabel("Precision / Recall")
    axes[0, 1].legend(fontsize=7)
    axes[0, 1].grid(True, alpha=0.3)

    axes[1, 0].plot(x, lead_df["PR_AUC_Fog"], "o-", label="Model PR-AUC Fog")
    axes[1, 0].plot(x, lead_df["PR_AUC_Mist"], "s-", label="Model PR-AUC Mist")
    axes[1, 0].set_ylabel("PR-AUC")
    axes[1, 0].set_xlabel("Lead time (h)")
    axes[1, 0].legend(fontsize=8)
    axes[1, 0].grid(True, alpha=0.3)

    axes[1, 1].plot(x, lead_df["low_vis_precision"], "o-", label="Model Low-vis precision")
    axes[1, 1].plot(x, lead_df["false_positive_rate"], "s-", label="Model FPR (Clear)")
    axes[1, 1].plot(xi, ifs_df["low_vis_precision"], "o--", label="IFS Low-vis precision")
    axes[1, 1].plot(xi, ifs_df["false_positive_rate"], "s--", label="IFS FPR (Clear)")
    axes[1, 1].set_ylabel("Rate")
    axes[1, 1].set_xlabel("Lead time (h)")
    axes[1, 1].legend(fontsize=8)
    axes[1, 1].grid(True, alpha=0.3)

    fig.suptitle("48h PM10 model vs IFS (0-48h)")
    fig.tight_layout()
    save_figure(fig, os.path.join(output_dir, "fig_metrics_vs_lead_with_ifs.png"))
    plt.show()
elif ifs_df.empty:
    print("[IFS] 未生成可用逐时对比结果，因此未叠加到图中。")

ValueError: All arrays must be of the same length

In [ ]:
# ========= 模型 vs IFS：按 lead 的严格比较（修正版，适配二维 valid_time） =========
import os
import numpy as np
import pandas as pd
import xarray as xr
import matplotlib.pyplot as plt

# 依赖前面 notebook 已生成的变量：
# meta_sub, y_cls, compute_rare_event_report, fog_th, mist_th, output_dir, ifs_all_leads_nc
required_vars = ["meta_sub", "y_cls", "compute_rare_event_report", "fog_th", "mist_th", "output_dir", "ifs_all_leads_nc"]
missing = [v for v in required_vars if v not in globals()]
if missing:
    raise RuntimeError(f"请先运行前面的模型评估单元，缺少变量: {missing}")

def normalize_station_id(values):
    s = pd.Series(values, dtype="string").str.strip()
    s = s.str.replace(r"\.0$", "", regex=True)
    s = s.str.replace(r"^0+(?=\d)", "", regex=True)
    return s.fillna("")

def _vis_to_cls_scalar(vi):
    if not np.isfinite(vi):
        return -1
    if vi < 500:
        return 0
    if vi < 1000:
        return 1
    return 2

def build_ifs_long_table(ifs_nc):
    ds_ifs = xr.open_dataset(ifs_nc)
    try:
        if "VIS_ifs" not in ds_ifs:
            raise ValueError("IFS 文件中未找到 VIS_ifs")

        vis_ifs = np.asarray(ds_ifs["VIS_ifs"].values)
        if vis_ifs.ndim != 3:
            raise ValueError(f"VIS_ifs 必须是 3D (record, lead_hour, station)，实际是 {vis_ifs.shape}")

        n_rec, n_lead, n_st = vis_ifs.shape

        if "lead_hour" not in ds_ifs.coords:
            raise ValueError("IFS 文件缺少 lead_hour 坐标")
        lead_vals = np.asarray(ds_ifs["lead_hour"].values).astype(np.int32)
        if lead_vals.shape[0] != n_lead:
            raise ValueError(f"lead_hour 长度 {lead_vals.shape[0]} 与 n_lead={n_lead} 不一致")

        if "station" not in ds_ifs.coords:
            raise ValueError("IFS 文件缺少 station 坐标")
        stations = normalize_station_id(ds_ifs["station"].values)
        if len(stations) != n_st:
            raise ValueError(f"station 长度 {len(stations)} 与 n_st={n_st} 不一致")

        # 关键修正：不要对二维数组直接 pd.to_datetime
        if "valid_time" in ds_ifs.coords:
            valid_time_grid = np.asarray(ds_ifs["valid_time"].values).astype("datetime64[ns]")
            if valid_time_grid.shape != (n_rec, n_lead):
                raise ValueError(
                    f"valid_time shape={valid_time_grid.shape}, expected {(n_rec, n_lead)}"
                )
        elif "forecast_reference_time" in ds_ifs.coords:
            fcref = np.asarray(ds_ifs["forecast_reference_time"].values).astype("datetime64[ns]")
            if fcref.shape[0] != n_rec:
                raise ValueError(
                    f"forecast_reference_time len={fcref.shape[0]}, expected {n_rec}"
                )
            valid_time_grid = fcref[:, None] + lead_vals[None, :].astype("timedelta64[h]")
        else:
            raise ValueError("IFS 文件既没有 valid_time，也没有 forecast_reference_time")

        # 展平
        valid_time_long = np.repeat(valid_time_grid.reshape(-1), n_st)
        lead_idx = np.tile(np.repeat(np.arange(n_lead), n_st), n_rec)
        st_idx = np.tile(np.arange(n_st), n_rec * n_lead)

        df = pd.DataFrame(
            {
                "t": pd.to_datetime(valid_time_long),
                "lead_hour": lead_vals[lead_idx].astype(np.int32),
                "station_id_norm": stations[st_idx].astype(str),
                "vis_ifs": vis_ifs.reshape(-1).astype(np.float64),
            }
        )

        df = df[np.isfinite(df["vis_ifs"])].copy()
        df["ifs_cls"] = df["vis_ifs"].map(_vis_to_cls_scalar).astype(np.int32)
        df = df[df["ifs_cls"] >= 0].copy()

        # exact key
        df["join_key"] = (
            df["t"].dt.strftime("%Y-%m-%d %H:%M:%S")
            + "__"
            + df["station_id_norm"]
            + "__"
            + df["lead_hour"].astype(str)
        )

        n_dup = int(df["join_key"].duplicated().sum())
        print(f"[IFS] rows={len(df)}, duplicated exact keys={n_dup}")
        if n_dup > 0:
            df = df.drop_duplicates("join_key", keep="first").copy()
            print(f"[IFS] after drop_duplicates -> rows={len(df)}")

        return df.reset_index(drop=True)

    finally:
        ds_ifs.close()

# ---------------- 模型侧长表 ----------------
df_model = meta_sub[["time", "lead_hour", "station_id_norm"]].copy()
df_model["t"] = pd.to_datetime(df_model["time"])
df_model["lead_hour"] = np.rint(pd.to_numeric(df_model["lead_hour"], errors="coerce")).astype("Int64")
df_model["station_id_norm"] = normalize_station_id(df_model["station_id_norm"])
df_model["obs_cls"] = np.asarray(y_cls).astype(np.int64)
df_model = df_model[df_model["lead_hour"].notna()].copy()
df_model["lead_hour"] = df_model["lead_hour"].astype(np.int32)

df_model["join_key"] = (
    df_model["t"].dt.strftime("%Y-%m-%d %H:%M:%S")
    + "__"
    + df_model["station_id_norm"]
    + "__"
    + df_model["lead_hour"].astype(str)
)

print(f"[MODEL] rows={len(df_model)}, unique exact keys={df_model['join_key'].nunique()}")
print(f"[MODEL] lead range={df_model['lead_hour'].min()}..{df_model['lead_hour'].max()}")

# ---------------- IFS 侧长表 ----------------
if not os.path.exists(ifs_all_leads_nc):
    raise FileNotFoundError(f"IFS 文件不存在: {ifs_all_leads_nc}")

df_ifs = build_ifs_long_table(ifs_all_leads_nc)
print(f"[IFS] unique exact keys={df_ifs['join_key'].nunique()}")
print(f"[IFS] lead range={df_ifs['lead_hour'].min()}..{df_ifs['lead_hour'].max()}")

# ---------------- exact join ----------------
merged = df_model.merge(
    df_ifs[["join_key", "vis_ifs", "ifs_cls"]],
    on="join_key",
    how="left"
)

exact_hits = int(merged["ifs_cls"].notna().sum())
print(f"[JOIN] exact key matches: {exact_hits} / {len(merged)} ({exact_hits / max(len(merged),1):.2%})")

if exact_hits == 0:
    raise RuntimeError("模型样本与 IFS 在 (valid_time, lead_hour, station) 上没有 exact overlap，请检查文件或时间定义。")

merged_ok = merged[merged["ifs_cls"].notna()].copy()
merged_ok["ifs_cls"] = merged_ok["ifs_cls"].astype(np.int32)
merged_ok.to_csv(os.path.join(output_dir, "ifs_model_joined_samples.csv"), index=False)
print(f"[JOIN] usable rows: {len(merged_ok)}")

# ---------------- 按 lead 计算 IFS 指标 ----------------
rows = []
for h, sub in merged_ok.groupby("lead_hour"):
    if len(sub) < 50:
        continue

    y_t = sub["obs_cls"].values.astype(np.int64)
    y_p = sub["ifs_cls"].values.astype(np.int64)

    rep = compute_rare_event_report(
        np.full((len(sub), 3), 1.0 / 3.0),
        y_t,
        fog_th,
        mist_th,
        pred=y_p,
    )

    row = {"lead_hour": int(h), "n": int(len(sub))}
    for k in [
        "Fog_CSI", "Fog_R", "Fog_P",
        "Mist_CSI", "Mist_P", "Mist_R",
        "low_vis_precision", "false_positive_rate"
    ]:
        row[k] = float(rep.get(k, np.nan))
    rows.append(row)

ifs_df = pd.DataFrame(rows).sort_values("lead_hour").reset_index(drop=True)
ifs_csv = os.path.join(output_dir, "ifs_metrics_by_lead_hour.csv")
ifs_df.to_csv(ifs_csv, index=False)
print("[IFS] saved:", ifs_csv)
display(ifs_df.head(20))

# ---------------- 与模型 lead_df 合并 ----------------
if "lead_df" in globals() and isinstance(lead_df, pd.DataFrame) and len(lead_df):
    cmp_df = lead_df.merge(
        ifs_df,
        on="lead_hour",
        how="inner",
        suffixes=("_model", "_ifs")
    ).sort_values("lead_hour").reset_index(drop=True)

    cmp_csv = os.path.join(output_dir, "model_vs_ifs_metrics_by_lead_hour.csv")
    cmp_df.to_csv(cmp_csv, index=False)
    print("[CMP] saved:", cmp_csv)
    display(cmp_df.head(20))

    fig, axes = plt.subplots(2, 2, figsize=(14, 10), constrained_layout=True)

    x = cmp_df["lead_hour"].values

    axes[0, 0].plot(x, cmp_df["Fog_CSI_model"], "o-", label="Model Fog CSI")
    axes[0, 0].plot(x, cmp_df["Fog_CSI_ifs"], "o--", label="IFS Fog CSI")
    axes[0, 0].set_title("Fog CSI vs Lead")
    axes[0, 0].grid(True, alpha=0.3)
    axes[0, 0].legend()

    axes[0, 1].plot(x, cmp_df["Fog_R_model"], "o-", label="Model Fog Recall")
    axes[0, 1].plot(x, cmp_df["Fog_R_ifs"], "o--", label="IFS Fog Recall")
    axes[0, 1].set_title("Fog Recall vs Lead")
    axes[0, 1].grid(True, alpha=0.3)
    axes[0, 1].legend()

    axes[1, 0].plot(x, cmp_df["Mist_P_model"], "o-", label="Model Mist Precision")
    axes[1, 0].plot(x, cmp_df["Mist_P_ifs"], "o--", label="IFS Mist Precision")
    axes[1, 0].set_title("Mist Precision vs Lead")
    axes[1, 0].grid(True, alpha=0.3)
    axes[1, 0].legend()

    axes[1, 1].plot(x, cmp_df["Mist_R_model"], "o-", label="Model Mist Recall")
    axes[1, 1].plot(x, cmp_df["Mist_R_ifs"], "o--", label="IFS Mist Recall")
    axes[1, 1].set_title("Mist Recall vs Lead")
    axes[1, 1].grid(True, alpha=0.3)
    axes[1, 1].legend()

    fig_path = os.path.join(output_dir, "fig_model_vs_ifs_metrics_by_lead.png")
    plt.savefig(fig_path, dpi=160, bbox_inches="tight")
    plt.show()
    print("[FIG] saved:", fig_path)
else:
    print("[WARN] lead_df 不存在，已生成 ifs_metrics_by_lead_hour.csv，但未合并模型结果。")

[MODEL] rows=1876216, unique exact keys=1876216
[MODEL] lead range=12..24
[IFS] rows=76203477, duplicated exact keys=0


KeyboardInterrupt: 